In [1]:
"""
CGNT Stage 3 — Causal Intervention Engine (CIE) + Joint Fine-tuning
=====================================================================
Adds physics-based causal reasoning on top of the Stage 2 residual GCN.

The CIE computes a Causal Inconsistency Score (CIS) for each sensor:
    CIS_i = || residual - H * H†_i * residual_i ||

Intuition: "If sensor i were the root cause of the observed residual,
what pattern would DC power flow physics predict across all 54 sensors?
How inconsistent is that prediction with the actual residual?"

- For PRIMARY sensors: CIS is LOW  — the residual pattern is exactly what
  physics predicts if that bus were attacked. Physics is consistent.
- For SECONDARY sensors: CIS is HIGH — the residual at this sensor is a
  propagation effect; if we pretend it's the cause, the physics prediction
  does not match the full residual pattern.
- For CLEAN sensors: CIS is near zero (residual_i ≈ 0, so H†_i * residual_i ≈ 0)

This CIS score is appended to the GCN node embedding before the classifier.
The joint fine-tuning unfreezes all components and trains end-to-end.

Inputs required (same folder):
    transformer_best.pt       Stage 1 transformer weights
    transformer_norm.npz      Stage 1 normalization (mu, sig in MW)
    residual_gcn_best.pt      Stage 2 GCN weights
    H_true_ieee14.npy         Jacobian matrix (54 x 13)
    cgnt_dataset_v3.csv       Rebalanced attack dataset
    hourly_dataset.xlsx        Benign hourly sensor data

Outputs:
    cgnt_full_best.pt          Best joint model weights
    stage3_test_results.npz    Final predictions + labels

Author: RAAGAVAN CGNT Research
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch_geometric.nn import GATv2Conv
from torch_geometric.data import Data, DataLoader
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import train_test_split
import math
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# 0.  CONFIG
# ─────────────────────────────────────────────
HOURLY_PATH      = "hourly_dataset.xlsx"
DATASET_PATH     = "cgnt_dataset_v3.csv"
H_MATRIX_PATH    = "H_true_ieee14.npy"
TRANSFORMER_PT   = "transformer_best.pt"
TRANSFORMER_NORM = "transformer_norm.npz"
GCN_PT           = "residual_gcn_best.pt"

WINDOW       = 48
BASE_MVA     = 100.0
BATCH_SIZE   = 32        # smaller batch for joint training stability
EPOCHS       = 100
LR           = 2e-4      # lower LR for fine-tuning
WARMUP_EPOCHS = 3
HIDDEN_DIM   = 64
NUM_HEADS    = 4
DROPOUT      = 0.3
WEIGHT_DECAY = 3e-4
EARLY_STOP   = 20
SENSOR_EMBED_DIM = 8
RANDOM_SEED  = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print(f"Device: {DEVICE}")


# ─────────────────────────────────────────────
# 1.  TOPOLOGY & CONSTANTS
# ─────────────────────────────────────────────
LINES = [
    (1,2),(1,5),(2,3),(2,4),(2,5),(3,4),(4,5),(4,7),(4,9),
    (5,6),(6,11),(6,12),(6,13),(7,8),(7,9),(9,10),(9,14),
    (10,11),(12,13),(13,14)
]
SENSOR_TYPE_IDS  = np.array([0]*20 + [1]*20 + [2]*14, dtype=np.int64)
NUM_SENSOR_TYPES = 3


def build_sensor_graph():
    edges = set()
    def add_edge(i, j):
        if i != j: edges.add((min(i,j), max(i,j)))
    for line_idx, (f_bus, t_bus) in enumerate(LINES):
        fwd  = line_idx;      rev  = line_idx + 20
        fbus = 40+(f_bus-1);  tbus = 40+(t_bus-1)
        add_edge(fwd, rev);   add_edge(fwd, fbus); add_edge(fwd, tbus)
        add_edge(rev, fbus);  add_edge(rev, tbus); add_edge(fbus, tbus)
    src, dst = [], []
    for i, j in edges:
        src.extend([i, j]); dst.extend([j, i])
    return torch.tensor([src, dst], dtype=torch.long)


# ─────────────────────────────────────────────
# 2.  CAUSAL INTERVENTION ENGINE
# ─────────────────────────────────────────────

class CausalInterventionEngine:
    """
    Computes Causal Inconsistency Scores (CIS) for each sensor.

    For sensor i:
        CIS_i = || residual - H * H†_col_i * residual_i ||_2

    where H†_col_i is the i-th column of the Moore-Penrose pseudoinverse,
    normalized so that H * H†_col_i * residual_i reproduces a full-system
    residual pattern consistent with sensor i being the root cause.

    Precomputes H * H†  (54x54 projection matrix) once for efficiency.
    """

    def __init__(self, H, device):
        H = H.astype(np.float64)   # use float64 for numerical stability
        # H†  via SVD (numerically stable pseudoinverse)
        U, s, Vt = np.linalg.svd(H, full_matrices=False)
        tol = max(H.shape) * np.finfo(float).eps * s[0]
        s_inv = np.where(s > tol, 1.0/s, 0.0)
        H_pinv = (Vt.T * s_inv) @ U.T     # (13, 54)

        # Projection matrix P = H @ H†  shape (54, 54)
        # P[i, :] = the full residual pattern predicted if sensor i is root cause
        # with unit perturbation
        P = H @ H_pinv                     # (54, 54)

        self.P = torch.tensor(P, dtype=torch.float32).to(device)
        self.device = device
        print(f"CIE initialized: P shape={P.shape}, "
              f"rank≈{np.linalg.matrix_rank(H)}")

    @torch.no_grad()
    def compute_cis(self, residuals_norm):
        """
        Parameters
        ----------
        residuals_norm : Tensor (N, 54)
            Normalized residuals from transformer

        Returns
        -------
        cis : Tensor (N, 54)
            Causal Inconsistency Score per sensor per sample.
            Low CIS  -> sensor is consistent with being root cause (PRIMARY)
            High CIS -> sensor shows propagation, not root cause (SECONDARY/CLEAN)
        """
        # For each sensor i:
        #   predicted_pattern_i = P[:, i] * residual_i   shape (N, 54)
        #   CIS_i = || residual - predicted_pattern_i ||

        N, S = residuals_norm.shape    # (N, 54)
        r = residuals_norm             # (N, 54)

        # P[:, i] * r[:, i]  for all i simultaneously
        # r[:, i] : (N,)  ->  r[:, i].unsqueeze(1) : (N, 1)
        # P[:, i] : (54,) ->  P[:, i].unsqueeze(0) : (1, 54)
        # outer product per sample per sensor: (N, 54, 54)
        # but we only need the norm, so we can be smarter:

        # predicted[n, :, i] = P[:, i] * r[n, i]
        # shape: (N, 54, 54)  — axis2 = which sensor is the "cause"
        predicted = self.P.unsqueeze(0) * r.unsqueeze(1)  # (N, 54, 54)
        # predicted[:, :, i] = predicted residual pattern if sensor i is cause

        # CIS[:, i] = || r - predicted[:, :, i] ||
        # r: (N, 54) -> unsqueeze(2) -> (N, 54, 1) broadcast with (N, 54, 54)
        diff = r.unsqueeze(2) - predicted          # (N, 54, 54)
        cis  = diff.norm(dim=1)                    # (N, 54)

        # Normalize CIS to [0, 1] per sample for stable GCN input
        cis_max = cis.max(dim=1, keepdim=True).values.clamp(min=1e-8)
        cis_norm = cis / cis_max

        return cis_norm   # (N, 54)  in [0, 1]


# ─────────────────────────────────────────────
# 3.  MODELS (copied from Stage 1 & 2)
# ─────────────────────────────────────────────

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=200, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float()
                        * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


class TemporalTransformer(nn.Module):
    def __init__(self, d_model=54, nhead=6, num_layers=2,
                 lstm_hidden=128, dropout=0.1):
        super().__init__()
        self.pos_enc = PositionalEncoding(d_model, dropout=dropout)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_model*4,
            dropout=dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(enc_layer,
                                                  num_layers=num_layers)
        self.lstm1 = nn.LSTM(d_model, lstm_hidden, batch_first=True)
        self.lstm2 = nn.LSTM(lstm_hidden, lstm_hidden, batch_first=True)
        self.head  = nn.Sequential(
            nn.Linear(lstm_hidden, lstm_hidden),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(lstm_hidden, d_model))

    def forward(self, x):
        x = self.pos_enc(x)
        x = self.transformer(x)
        x, _ = self.lstm1(x)
        x, _ = self.lstm2(x)
        return self.head(x[:, -1, :])


class CausalGCN(nn.Module):
    """
    GCN augmented with CIS scores from the Causal Intervention Engine.
    Input per node: residual(1) + H_row(13) + type_embed(8) + CIS(1) = 23 dims
    The CIS feature is the key addition over Stage 2.
    """

    def __init__(self, h_dim=13, embed_dim=8, hidden_dim=64,
                 num_heads=4, num_classes=3, dropout=0.3):
        super().__init__()
        self.dropout    = dropout
        in_dim = 1 + h_dim + embed_dim + 1   # +1 for CIS score = 23

        self.type_embed = nn.Embedding(NUM_SENSOR_TYPES, embed_dim)
        self.input_proj = nn.Linear(in_dim, hidden_dim)

        self.conv1 = GATv2Conv(in_dim, hidden_dim, heads=num_heads,
                               dropout=dropout, concat=True)
        self.conv2 = GATv2Conv(hidden_dim*num_heads, hidden_dim,
                               heads=num_heads, dropout=dropout, concat=True)
        self.conv3 = GATv2Conv(hidden_dim*num_heads, hidden_dim,
                               heads=1, dropout=dropout, concat=False)

        self.bn1 = nn.BatchNorm1d(hidden_dim * num_heads)
        self.bn2 = nn.BatchNorm1d(hidden_dim * num_heads)
        self.bn3 = nn.BatchNorm1d(hidden_dim)

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x, edge_index, sensor_type):
        type_emb = self.type_embed(sensor_type)
        x_full   = torch.cat([x, type_emb], dim=1)    # (N, 23)
        residual = F.relu(self.input_proj(x_full))     # (N, 64)

        h = F.elu(self.bn1(self.conv1(x_full, edge_index)))
        h = F.dropout(h, p=self.dropout, training=self.training)
        h = F.elu(self.bn2(self.conv2(h, edge_index)))
        h = F.dropout(h, p=self.dropout, training=self.training)
        h = F.elu(self.bn3(self.conv3(h, edge_index)))

        h = torch.cat([h, residual], dim=1)
        return self.classifier(h)


# ─────────────────────────────────────────────
# 4.  DATASET BUILDER
# ─────────────────────────────────────────────

def build_dataset(transformer, cie, mu, sig, device):
    print("\nLoading benign hourly data...")
    df_h = pd.read_excel(HOURLY_PATH)
    cols  = [f'line_{i}' for i in range(1,41)] + \
            [f'Cons_{i}'  for i in range(1,15)]
    benign_mw   = df_h[cols].values.astype(np.float32)
    benign_norm = (benign_mw - mu) / sig

    print("Loading attack dataset...")
    df_a = pd.read_csv(DATASET_PATH)
    attack_pu  = df_a[[f's{i}'        for i in range(1,55)]].values.astype(np.float32)
    labels     = df_a[[f'label_s{i}'  for i in range(1,55)]].values.astype(np.int64)
    k_values   = df_a['k_value'].values

    N   = len(attack_pu)
    rng = np.random.default_rng(RANDOM_SEED)
    t_indices = rng.integers(WINDOW, len(benign_mw), size=N)

    print(f"Computing residuals + CIS for {N} samples...")
    residuals_all = np.zeros((N, 54), dtype=np.float32)
    cis_all       = np.zeros((N, 54), dtype=np.float32)

    transformer.eval()
    bs = 256
    with torch.no_grad():
        for start in range(0, N, bs):
            end     = min(start + bs, N)
            batch_t = t_indices[start:end]

            contexts = np.stack([benign_norm[t-WINDOW:t] for t in batch_t])
            ctx_t    = torch.tensor(contexts, dtype=torch.float32).to(device)
            pred_norm = transformer(ctx_t).cpu().numpy()

            benign_t  = np.stack([benign_norm[t] for t in batch_t])
            delta_mw  = attack_pu[start:end] * BASE_MVA
            delta_norm = np.clip(delta_mw / sig, -20.0, 20.0)

            observed_norm         = benign_t + delta_norm
            residuals_all[start:end] = observed_norm - pred_norm

            if (start // bs) % 10 == 0:
                print(f"  {end}/{N}...")

    # Compute CIS in batches on GPU
    print("Computing CIS scores...")
    r_tensor = torch.tensor(residuals_all, dtype=torch.float32).to(device)
    cis_bs   = 512
    for start in range(0, N, cis_bs):
        end = min(start + cis_bs, N)
        cis_all[start:end] = cie.compute_cis(
            r_tensor[start:end]).cpu().numpy()

    print(f"Done.")
    print(f"  Residual range: [{residuals_all.min():.3f}, {residuals_all.max():.3f}]")
    print(f"  CIS at PRIMARY sensors (mean):   "
          f"{cis_all[labels==1].mean():.4f}")
    print(f"  CIS at SECONDARY sensors (mean): "
          f"{cis_all[labels==2].mean():.4f}")
    print(f"  CIS at CLEAN sensors (mean):     "
          f"{cis_all[labels==0].mean():.4f}")

    return residuals_all, cis_all, labels, k_values


def make_pyg_data_list(residuals, cis_scores, labels,
                        edge_index, H_features):
    sensor_type_ids = torch.tensor(SENSOR_TYPE_IDS, dtype=torch.long)
    H_feat = torch.tensor(H_features, dtype=torch.float32)
    data_list = []
    for i in range(len(residuals)):
        r   = torch.tensor(residuals[i],  dtype=torch.float32).unsqueeze(1)
        cis = torch.tensor(cis_scores[i], dtype=torch.float32).unsqueeze(1)
        x   = torch.cat([r, H_feat, cis], dim=1)   # (54, 15)
        y   = torch.tensor(labels[i], dtype=torch.long)
        data_list.append(Data(x=x, edge_index=edge_index,
                              y=y, sensor_type=sensor_type_ids))
    return data_list


def compute_class_weights(Y):
    flat    = Y.ravel()
    counts  = np.bincount(flat, minlength=3).astype(float)
    weights = 1.0 / (np.log1p(counts) + 1e-8)
    weights = weights / weights.sum() * 3
    w = torch.tensor(weights, dtype=torch.float32)
    print(f"Class counts  — CLEAN:{counts[0]:.0f}  "
          f"PRIMARY:{counts[1]:.0f}  SECONDARY:{counts[2]:.0f}")
    print(f"Class weights — CLEAN:{w[0]:.3f}  "
          f"PRIMARY:{w[1]:.3f}  SECONDARY:{w[2]:.3f}")
    return w


# ─────────────────────────────────────────────
# 5.  TRAIN / EVAL
# ─────────────────────────────────────────────

def get_lr(epoch, warmup, total, base_lr):
    if epoch < warmup:
        return base_lr * (epoch+1) / warmup
    progress = (epoch-warmup) / max(total-warmup, 1)
    return base_lr * 0.5 * (1 + math.cos(math.pi * progress))


def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total = 0.0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out  = model(batch.x, batch.edge_index, batch.sensor_type)
        loss = criterion(out, batch.y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total, preds, labels = 0.0, [], []
    for batch in loader:
        batch = batch.to(device)
        out   = model(batch.x, batch.edge_index, batch.sensor_type)
        total += criterion(out, batch.y).item()
        preds.append(out.argmax(dim=1).cpu().numpy())
        labels.append(batch.y.cpu().numpy())
    return (total/len(loader),
            np.concatenate(preds),
            np.concatenate(labels))


def compute_racc(preds, labels, n=54):
    m = len(preds) // n
    return (preds.reshape(m, n) == labels.reshape(m, n)).all(axis=1).mean()


def print_metrics(preds, labels, name, prev_racc=None):
    print(f"\n{'='*52}\n  {name}\n{'='*52}")
    print(classification_report(labels, preds,
          target_names=['CLEAN','PRIMARY','SECONDARY'],
          digits=4, zero_division=0))
    racc = compute_racc(preds, labels)
    print(f"  RACC : {racc:.4f}  ({100*racc:.2f}%)")
    if prev_racc:
        print(f"  Stage 2 RACC: {prev_racc:.4f}  "
              f"(improvement: {racc-prev_racc:+.4f})")
    print(f"  XTM  : 0.9299  (92.99%)")
    print(f"  Gap  : {0.9299-racc:+.4f}")
    print(f"{'='*52}")
    return racc


# ─────────────────────────────────────────────
# 6.  MAIN
# ─────────────────────────────────────────────

def main():
    print("\n" + "="*60)
    print("  CGNT Stage 3 — CIE + Joint Fine-tuning")
    print("  Physics-informed causal attribution")
    print("="*60)

    # Load normalization
    norm = np.load(TRANSFORMER_NORM)
    mu   = norm['mu'].astype(np.float32)
    sig  = norm['sig'].astype(np.float32)

    # Load frozen transformer (for residual generation only)
    transformer = TemporalTransformer().to(DEVICE)
    transformer.load_state_dict(
        torch.load(TRANSFORMER_PT, map_location=DEVICE))
    transformer.eval()
    for p in transformer.parameters():
        p.requires_grad = False
    print(f"Transformer loaded [FROZEN for dataset building]")

    # Load H matrix and build CIE
    H_raw      = np.load(H_MATRIX_PATH).astype(np.float32)
    H_norm     = (H_raw - H_raw.mean(0)) / (H_raw.std(0) + 1e-8)
    cie        = CausalInterventionEngine(H_raw, DEVICE)
    edge_index = build_sensor_graph()

    # Build dataset with residuals + CIS
    residuals, cis_scores, labels, k_values = build_dataset(
        transformer, cie, mu, sig, DEVICE)

    # Split (same seed as Stage 2 for fair comparison)
    idx = np.arange(len(residuals))
    idx_train, idx_temp = train_test_split(idx, test_size=0.30,
                          random_state=RANDOM_SEED, stratify=k_values)
    idx_val, idx_test   = train_test_split(idx_temp, test_size=0.50,
                          random_state=RANDOM_SEED,
                          stratify=k_values[idx_temp])
    print(f"\nSplit — Train:{len(idx_train)}  "
          f"Val:{len(idx_val)}  Test:{len(idx_test)}")

    ei_cpu = edge_index.cpu()
    train_data = make_pyg_data_list(residuals[idx_train], cis_scores[idx_train],
                                    labels[idx_train], ei_cpu, H_norm)
    val_data   = make_pyg_data_list(residuals[idx_val],   cis_scores[idx_val],
                                    labels[idx_val],   ei_cpu, H_norm)
    test_data  = make_pyg_data_list(residuals[idx_test],  cis_scores[idx_test],
                                    labels[idx_test],  ei_cpu, H_norm)

    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_data,   batch_size=BATCH_SIZE, shuffle=False)
    test_loader  = DataLoader(test_data,  batch_size=BATCH_SIZE, shuffle=False)

    class_weights = compute_class_weights(labels[idx_train]).to(DEVICE)
    criterion     = nn.CrossEntropyLoss(weight=class_weights)

    # Initialize CausalGCN and load Stage 2 weights where possible
    model = CausalGCN(
        h_dim=13, embed_dim=SENSOR_EMBED_DIM,
        hidden_dim=HIDDEN_DIM, num_heads=NUM_HEADS,
        num_classes=3, dropout=DROPOUT
    ).to(DEVICE)

    # Load Stage 2 weights (all layers match except input_proj and conv1
    # which now have 23 input dims instead of 22)
    stage2_state = torch.load(GCN_PT, map_location=DEVICE)
    model_state  = model.state_dict()
    transferred, skipped = 0, 0
    for k, v in stage2_state.items():
        if k in model_state and model_state[k].shape == v.shape:
            model_state[k] = v
            transferred += 1
        else:
            skipped += 1
    model.load_state_dict(model_state)
    print(f"\nStage 2 weights transferred: {transferred} layers  "
          f"(skipped {skipped} due to shape change from CIS dimension)")
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

    optimizer = torch.optim.Adam(model.parameters(),
                                  lr=LR, weight_decay=WEIGHT_DECAY)

    # Training
    print(f"\nFine-tuning for up to {EPOCHS} epochs...")
    print(f"{'Epoch':>6} {'LR':>10} {'Train Loss':>12} "
          f"{'Val Loss':>10} {'Val RACC':>10}")
    print("-" * 55)

    best_val_loss, best_epoch, patience = float('inf'), 0, 0

    for epoch in range(1, EPOCHS+1):
        lr = get_lr(epoch-1, WARMUP_EPOCHS, EPOCHS, LR)
        for pg in optimizer.param_groups: pg['lr'] = lr

        train_loss = train_epoch(model, train_loader,
                                  optimizer, criterion, DEVICE)
        val_loss, val_preds, val_labels = evaluate(
            model, val_loader, criterion, DEVICE)
        val_racc = compute_racc(val_preds, val_labels)

        if epoch % 5 == 0 or epoch == 1:
            print(f"{epoch:>6} {lr:>10.6f} {train_loss:>12.4f} "
                  f"{val_loss:>10.4f} {val_racc:>10.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss; best_epoch = epoch; patience = 0
            torch.save(model.state_dict(), "cgnt_full_best.pt")
        else:
            patience += 1
            if patience >= EARLY_STOP:
                print(f"\nEarly stopping at epoch {epoch} "
                      f"(best: epoch {best_epoch})")
                break

    # Final evaluation
    print(f"\nLoading best model from epoch {best_epoch}...")
    model.load_state_dict(
        torch.load("cgnt_full_best.pt", map_location=DEVICE))

    _, val_preds,  val_labels  = evaluate(model, val_loader,
                                           criterion, DEVICE)
    _, test_preds, test_labels = evaluate(model, test_loader,
                                           criterion, DEVICE)

    stage2_racc = 0.9193   # from Stage 2 test results
    print_metrics(val_preds,  val_labels,  "VALIDATION", stage2_racc)
    test_racc = print_metrics(test_preds, test_labels, "TEST", stage2_racc)

    # Per-k breakdown
    k_test = k_values[idx_test]
    p2d    = test_preds.reshape(-1, 54)
    l2d    = test_labels.reshape(-1, 54)
    print("\n=== PERFORMANCE BY k_value ===")
    stage2_k = {1:0.9893, 2:0.9200, 3:0.7867, 4:0.6133}
    for k in sorted(np.unique(k_test)):
        mask = k_test == k
        racc = compute_racc(p2d[mask].ravel(), l2d[mask].ravel())
        f1_p = f1_score(l2d[mask].ravel(), p2d[mask].ravel(),
                        labels=[1], average='macro', zero_division=0)
        f1_s = f1_score(l2d[mask].ravel(), p2d[mask].ravel(),
                        labels=[2], average='macro', zero_division=0)
        delta = racc - stage2_k.get(k, 0)
        print(f"  k={k}: RACC={racc:.4f} ({delta:+.4f} vs Stage2)  "
              f"PRIMARY_F1={f1_p:.4f}  SECONDARY_F1={f1_s:.4f}  (n={mask.sum()})")

    # Verify CIS is doing what we expect
    print("\n=== CIS VERIFICATION ===")
    print("(CIS should be LOW for PRIMARY, HIGH for SECONDARY)")
    cis_test = cis_scores[idx_test]
    l_test   = labels[idx_test]
    print(f"  Mean CIS at CLEAN    sensors: {cis_test[l_test==0].mean():.4f}")
    print(f"  Mean CIS at PRIMARY  sensors: {cis_test[l_test==1].mean():.4f}")
    print(f"  Mean CIS at SECONDARY sensors:{cis_test[l_test==2].mean():.4f}")

    # Save
    np.savez("stage3_test_results.npz",
             preds=test_preds, labels=test_labels,
             k_values=k_test, cis_scores=cis_test)
    print("\nSaved: stage3_test_results.npz")
    print("Saved: cgnt_full_best.pt")

    # Final comparison table
    print("\n" + "="*60)
    print("  FINAL COMPARISON TABLE")
    print("="*60)
    print(f"  {'Model':<35} {'RACC':>8}")
    print(f"  {'-'*43}")
    print(f"  {'XTM (Baul et al. 2023)':<35} {'0.9299':>8}")
    print(f"  {'CGNT Stage 2 (residual GCN)':<35} {'0.9193':>8}")
    print(f"  {'CGNT Stage 3 (+ CIE)':<35} {test_racc:>8.4f}")
    print(f"  {'-'*43}")
    if test_racc > 0.9299:
        print(f"  >> CGNT BEATS XTM by {test_racc-0.9299:.4f} RACC points")
    else:
        print(f"  >> Gap to XTM: {0.9299-test_racc:.4f}")
    print("="*60)


if __name__ == '__main__':
    main()

Device: cuda

  CGNT Stage 3 — CIE + Joint Fine-tuning
  Physics-informed causal attribution
Transformer loaded [FROZEN for dataset building]
CIE initialized: P shape=(54, 54), rank≈13

Loading benign hourly data...
Loading attack dataset...
Computing residuals + CIS for 10000 samples...
  256/10000...
  2816/10000...
  5376/10000...
  7936/10000...
Computing CIS scores...
Done.
  Residual range: [-20.755, 20.755]
  CIS at PRIMARY sensors (mean):   0.9507
  CIS at SECONDARY sensors (mean): 0.9643
  CIS at CLEAN sensors (mean):     0.9996

Split — Train:7000  Val:1500  Test:1500
Class counts  — CLEAN:268681  PRIMARY:80266  SECONDARY:29053
Class weights — CLEAN:0.903  PRIMARY:0.999  SECONDARY:1.098

Stage 2 weights transferred: 37 layers  (skipped 3 due to shape change from CIS dimension)
Model parameters: 189,083

Fine-tuning for up to 100 epochs...
 Epoch         LR   Train Loss   Val Loss   Val RACC
-------------------------------------------------------
     1   0.000067       1.5353